# Real-Time Crisis Detection System
## Notebook 05 — Credibility Scoring and Event Clustering
**Goal:** Score posts for credibility, embed with SBERT, build composite distance matrix, run DBSCAN, extract canonical event records. Save `clustered_events.csv` and `master_events.csv`.

In [1]:
import os, sys, warnings; warnings.filterwarnings('ignore')
IN_COLAB='google.colab' in sys.modules; IN_KAGGLE='KAGGLE_URL_BASE' in os.environ
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    PROJECT_DIR='/content/drive/MyDrive/10Academy/crisis-detection-system'
elif IN_KAGGLE: PROJECT_DIR='/kaggle/working/crisis-detection-system'
else: PROJECT_DIR=os.path.abspath('..')
DATA_PROCESSED=f'{PROJECT_DIR}/data/processed'
MODELS_DIR=f'{PROJECT_DIR}/models'
OUTPUTS_DIR=f'{PROJECT_DIR}/outputs'
UTILS_DIR=f'{PROJECT_DIR}/utils'
os.makedirs(OUTPUTS_DIR, exist_ok=True)
if UTILS_DIR not in sys.path: sys.path.insert(0, UTILS_DIR)
print(f' Ready. Project: {PROJECT_DIR}')

Mounted at /content/drive
✅ Ready. Project: /content/drive/MyDrive/10Academy/crisis-detection-system


In [2]:
!pip install -q sentence-transformers scikit-learn numpy

In [3]:
# & 2: Load and merge all upstream outputs
import pandas as pd, numpy as np, json, pickle, re
from tqdm import tqdm; tqdm.pandas()
import matplotlib.pyplot as plt

df_clf  = pd.read_csv(f'{DATA_PROCESSED}/classified_events.csv', index_col='post_id')
df_geo  = pd.read_csv(f'{DATA_PROCESSED}/geo_resolved.csv',      index_col='post_id')
df_temp = pd.read_csv(f'{DATA_PROCESSED}/temporal_resolved.csv', index_col='post_id')

# Merge all on post_id
df = df_clf.join(df_geo, how='left', rsuffix='_geo')
df = df.join(df_temp, how='left', rsuffix='_temp')

# Keep only classified posts for primary pipeline
df_work = df[df['classification_status'] == 'classified'].copy()
# Also exclude 'none' resolution status
if 'resolution_status' in df_work.columns:
    df_work = df_work[df_work['resolution_status'] != 'none']

print(f' Merged datasets. Working set: {len(df_work)} posts')
print(f'Columns available: {list(df.columns[:12])}...')

✅ Merged datasets. Working set: 28 posts
Columns available: ['source_platform', 'raw_text', 'text_cleaned', 'timestamp_raw', 'user_id', 'location_raw', 'geo_lat', 'geo_lon', 'media_present', 'engagement_score', 'engagement_score.1', 'label']...


In [4]:
# Engineer credibility features
from nlp_utils import compute_linguistic_uncertainty

# Feature 1: has_media (already in dataset)
# Check if 'media_present' column exists, otherwise default to 0.0
if 'media_present' in df_work.columns:
    df_work['has_media_f'] = df_work['media_present'].fillna(False).astype(float)
else:
    print("Warning: 'media_present' column not found. Setting 'has_media_f' to 0.0 for all posts.")
    df_work['has_media_f'] = 0.0

# Feature 2: normalized engagement score (log transform)
# Check if 'engagement_score' column exists, otherwise default to 0.
if 'engagement_score' in df_work.columns:
    df_work['engagement_log'] = np.log1p(df_work['engagement_score'].fillna(0))
else:
    print("Warning: 'engagement_score' column not found. Setting 'engagement_log' based on 0 engagement.")
    df_work['engagement_log'] = np.log1p(0) # log1p(0) is 0
max_eng = df_work['engagement_log'].max()
df_work['engagement_norm'] = df_work['engagement_log'] / max(max_eng, 1)

# Feature 3: news domain URL (credible source)
CREDIBLE_DOMAINS = ['kompas.com','detik.com','tribunnews.com','antaranews.com',
                    'cnnindonesia.com','tempo.co','liputan6.com','okezone.com',
                    'bbc.com','reuters.com','apnews.com']
def has_news_url(text):
    text_lower = str(text).lower()
    return float(any(d in text_lower for d in CREDIBLE_DOMAINS))

df_work['news_domain_url'] = df_work['text_cleaned'].apply(has_news_url)

# Feature 4: contains specific casualty/damage numbers
def has_specific_number(text):
    return float(bool(re.search(
        r'\d+\s*(orang|jiwa|rumah|korban|warga|kendaraan|person|people|house|building)',
        str(text), re.IGNORECASE
    )))
df_work['has_specific_number'] = df_work['text_cleaned'].apply(has_specific_number)

# Feature 5: linguistic uncertainty
if 'linguistic_uncertainty' not in df_work.columns:
    df_work['linguistic_uncertainty'] = df_work['text_cleaned'].progress_apply(
        lambda t: compute_linguistic_uncertainty(str(t))
    )

# Feature 6-7: geo and time confidence (from upstream)
df_work['geo_confidence']  = df_work['geo_confidence'].fillna(0.0)
df_work['time_confidence'] = df_work['time_confidence'].fillna(0.30)

print(' Credibility features engineered.')
CRED_FEATURES = ['has_media_f','engagement_norm','news_domain_url',
                 'has_specific_number','geo_confidence','time_confidence']
print(df_work[CRED_FEATURES].describe().round(3))

✅ Credibility features engineered.
       has_media_f  engagement_norm  news_domain_url  has_specific_number  \
count       28.000           28.000             28.0               28.000   
mean         0.357            0.157              0.0                0.107   
std          0.488            0.315              0.0                0.315   
min          0.000            0.000              0.0                0.000   
25%          0.000            0.000              0.0                0.000   
50%          0.000            0.000              0.0                0.000   
75%          1.000            0.000              0.0                0.000   
max          1.000            1.000              0.0                1.000   

       geo_confidence  time_confidence  
count          28.000             28.0  
mean            0.890              0.3  
std             0.148              0.0  
min             0.600              0.3  
25%             0.880              0.3  
50%             0.940    

In [5]:
# & 5: Linguistic uncertainty + Cross-source corroboration
print('Computing cross-source corroboration...')

# Convert timestamps to datetime
df_work['ts_dt'] = pd.to_datetime(
    df_work.get('timestamp_event_utc', df_work.get('timestamp_raw', '')),
    errors='coerce', utc=True
)

CORR_RADIUS_KM = 50
CORR_WINDOW_H  = 4

from geo_utils import haversine_distance
from datetime import timedelta

# Vectorized corroboration is expensive for large datasets
# We use a simplified implementation here
corr_scores = []
df_list = df_work.reset_index()

for i, row_i in tqdm(df_list.iterrows(), total=len(df_list), desc='Corroboration'):
    score = 0
    lat1, lon1 = row_i.get('geo_lat_resolved'), row_i.get('geo_lon_resolved')
    ts1 = row_i.get('ts_dt')
    src1 = row_i.get('source_platform', '')
    cls1 = row_i.get('predicted_class', '')

    corroborating_platforms = set()
    for j, row_j in df_list.iterrows():
        if i == j or row_j.get('source_platform') == src1:
            continue  # skip self and same platform
        if row_j.get('predicted_class') != cls1:
            continue
# Check time proximity
        ts2 = row_j.get('ts_dt')
        if pd.notna(ts1) and pd.notna(ts2):
            if abs((ts1-ts2).total_seconds()) > CORR_WINDOW_H*3600:
                continue
# Check geo proximity
        lat2, lon2 = row_j.get('geo_lat_resolved'), row_j.get('geo_lon_resolved')
        if pd.notna(lat1) and pd.notna(lat2) and pd.notna(lon1) and pd.notna(lon2):
            if haversine_distance(lat1, lon1, lat2, lon2) > CORR_RADIUS_KM:
                continue
        corroborating_platforms.add(row_j.get('source_platform', 'unknown'))
    score = min(len(corroborating_platforms), 2)
    corr_scores.append(score)

df_work['cross_corroboration'] = corr_scores
print(' Cross-source corroboration computed.')
print(f'Distribution: {pd.Series(corr_scores).value_counts().to_dict()}')

Computing cross-source corroboration...


Corroboration: 100%|██████████| 28/28 [00:00<00:00, 99.40it/s]

✅ Cross-source corroboration computed.
Distribution: {0: 26, 1: 2}


In [6]:
# Compute credibility score
# Weighted formula:
# credibility = 0.30×corroboration + 0.20×(1-uncertainty) + 0.15×media
# + 0.15×news_url + 0.10×engagement + 0.10×geo_confidence

W = {'corr':0.30, 'certainty':0.20, 'media':0.15, 'news':0.15,
     'engagement':0.10, 'geo':0.10}

corr_norm = df_work['cross_corroboration'] / 2.0  # max=2
certainty = 1.0 - df_work['linguistic_uncertainty']

df_work['credibility_score'] = (
    W['corr']       * corr_norm +
    W['certainty']  * certainty +
    W['media']      * df_work['has_media_f'] +
    W['news']       * df_work['news_domain_url'] +
    W['engagement'] * df_work['engagement_norm'] +
    W['geo']        * df_work['geo_confidence']
).clip(0.0, 1.0)

print(f' Credibility scores computed.')
print(f'Distribution: mean={df_work["credibility_score"].mean():.3f}, '
      f'std={df_work["credibility_score"].std():.3f}')

✅ Credibility scores computed.
Distribution: mean=0.366, std=0.093


In [7]:
# Apply credibility filter
def credibility_label(score):
    if score >= 0.5:   return 'credible'
    elif score >= 0.3: return 'uncertain'
    return 'low_credibility'

df_work['credibility_label'] = df_work['credibility_score'].apply(credibility_label)

# BMKG official always credible
df_work.loc[df_work.get('source_platform', '') == 'bmkg', 'credibility_label'] = 'credible'
df_work.loc[df_work.get('source_platform', '') == 'bmkg', 'credibility_score'] = 1.0

print('Credibility distribution:')
print(df_work['credibility_label'].value_counts().to_string())

# Only credible + uncertain proceed to clustering
df_cluster_input = df_work[df_work['credibility_label'] != 'low_credibility'].copy()
print(f'\nProceeding to clustering: {len(df_cluster_input)} posts')

Credibility distribution:
credibility_label
uncertain          14
low_credibility     9
credible            5

Proceeding to clustering: 19 posts


In [8]:
# Generate SBERT embeddings
from sentence_transformers import SentenceTransformer

SBERT_MODEL = 'paraphrase-multilingual-mpnet-base-v2'
EMBEDDINGS_CACHE = f'{MODELS_DIR}/sbert_embeddings_cache.pkl'

texts_to_embed = df_cluster_input['text_cleaned'].fillna('').astype(str).tolist()
post_ids_embed = df_cluster_input.index.tolist()

if os.path.exists(EMBEDDINGS_CACHE):
    with open(EMBEDDINGS_CACHE, 'rb') as f:
        cache = pickle.load(f)
    cached_ids = cache.get('post_ids', [])
# Check if cache matches current posts
    if set(cached_ids) == set(post_ids_embed):
        embeddings = cache['embeddings']
        print(f' Loaded embeddings from cache: {embeddings.shape}')
    else:
        embeddings = None
else:
    embeddings = None

if embeddings is None:
    print(f'Loading SBERT model: {SBERT_MODEL}...')
    sbert = SentenceTransformer(SBERT_MODEL)
    print(f'Encoding {len(texts_to_embed)} posts...')
    embeddings = sbert.encode(
        texts_to_embed, batch_size=32, show_progress_bar=True,
        normalize_embeddings=True
    )
    with open(EMBEDDINGS_CACHE, 'wb') as f:
        pickle.dump({'post_ids': post_ids_embed, 'embeddings': embeddings}, f)
    print(f' Embeddings generated and cached: {embeddings.shape}')

Loading SBERT model: paraphrase-multilingual-mpnet-base-v2...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 19 posts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings generated and cached: (19, 768)


In [9]:
# Build composite distance matrix (INLINE)
import numpy as np
from geo_utils import haversine_distance
from temporal_utils import temporal_proximity

timestamps_list = df_cluster_input['ts_dt'].tolist()
coords_list = list(zip(
    df_cluster_input['geo_lat_resolved'].tolist(),
    df_cluster_input['geo_lon_resolved'].tolist()
))

weights = {'semantic': 0.50, 'temporal': 0.30, 'spatial': 0.20}
N = len(embeddings)

# Semantic distance
# cosine similarity = dot product of unit vectors
norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norms = np.where(norms == 0, 1e-9, norms)
normed = embeddings / norms
cosine_sim = normed @ normed.T
semantic_dist = np.clip(1.0 - cosine_sim, 0.0, 1.0)

# Temporal distance
temporal_dist = np.full((N, N), 0.5)  # neutral default if unknown
for i in range(N):
    for j in range(i + 1, N):
        prox = temporal_proximity(timestamps_list[i], timestamps_list[j], window_hours=24)
        d = 1.0 - prox
        temporal_dist[i, j] = d
        temporal_dist[j, i] = d
np.fill_diagonal(temporal_dist, 0.0)

# Spatial distance
MAX_KM = 100.0
spatial_dist = np.full((N, N), 0.5)  # neutral if geo unknown
for i in range(N):
    for j in range(i + 1, N):
        lat1, lon1 = coords_list[i]
        lat2, lon2 = coords_list[j]
        if None in (lat1, lon1, lat2, lon2):
            d = 0.5  # unknown
        else:
            km = haversine_distance(lat1, lon1, lat2, lon2)
            d = min(1.0, km / MAX_KM)
        spatial_dist[i, j] = d
        spatial_dist[j, i] = d
np.fill_diagonal(spatial_dist, 0.0)

# Composite
dist_matrix = (
    weights['semantic']  * semantic_dist +
    weights['temporal']  * temporal_dist +
    weights['spatial']   * spatial_dist
)

print(f' Distance matrix shape: {dist_matrix.shape}')
print(f'   Mean distance: {dist_matrix.mean():.3f}  Max: {dist_matrix.max():.3f}')

✅ Distance matrix shape: (19, 19)
   Mean distance: 0.786  Max: 1.000


In [10]:
# Run DBSCAN event clustering
import numpy as np
try:
    from sklearn.cluster import DBSCAN
    _SKLEARN_AVAILABLE = True
except ImportError:
    _SKLEARN_AVAILABLE = False

EPS = 0.35
MIN_SAMPLES = 2

# Inlined run_dbscan function logic (from clustering_utils.py)
# This ensures the correct version is used and avoids module caching issues.
if not _SKLEARN_AVAILABLE:
    print("  scikit-learn not available. Returning all as noise.")
    cluster_labels = np.full(len(dist_matrix), -1)
else:
    db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES, metric='precomputed')
    cluster_labels = db.fit_predict(dist_matrix)

df_cluster_input['cluster_id'] = cluster_labels
df_cluster_input['is_singleton'] = (cluster_labels == -1)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_singletons = (cluster_labels == -1).sum()

print(f' DBSCAN complete.')
print(f'Events discovered: {n_clusters} clusters + {n_singletons} singletons')
print('\nCluster sizes:')
non_noise = df_cluster_input[df_cluster_input['cluster_id'] != -1]
if len(non_noise) > 0:
    print(non_noise.groupby('cluster_id').size().sort_values(ascending=False).head(10).to_string())

✅ DBSCAN complete.
Events discovered: 1 clusters + 17 singletons

Cluster sizes:
cluster_id
0    2


In [11]:
# Extract canonical event records
import numpy as np
from collections import Counter

# Inlined extract_canonical_event function logic (from clustering_utils.py)
def extract_canonical_event(cluster_df):
    if cluster_df.empty:
        return {}

# Event type: majority vote weighted by confidence
    event_type = cluster_df['predicted_class'].mode().iloc[0] \
        if not cluster_df['predicted_class'].isna().all() else 'unknown'

# Location: credibility-weighted centroid
    geo_rows = cluster_df.dropna(subset=['geo_lat_resolved', 'geo_lon_resolved'])
    if not geo_rows.empty:
        weights = geo_rows.get('geo_confidence', 1.0)
        if hasattr(weights, 'values'):
            weights = weights.fillna(0.5).values
        else:
            weights = np.ones(len(geo_rows))
        weights = np.where(weights == 0, 0.01, weights)
        event_lat = float(np.average(geo_rows['geo_lat_resolved'], weights=weights))
        event_lon = float(np.average(geo_rows['geo_lon_resolved'], weights=weights))
    else:
        event_lat = None
        event_lon = None

# Timestamp: median event time
# Use the pre-converted 'ts_dt' column which contains datetime objects
    timestamps = cluster_df['ts_dt'].dropna()
    event_timestamp = timestamps.median() if not timestamps.empty else None

# Representative post: highest credibility score
    cred_col = 'credibility_score' if 'credibility_score' in cluster_df.columns \
        else 'confidence_score'
    rep_row = cluster_df.loc[cluster_df[cred_col].idxmax()]
    representative_text = rep_row.get('text_cleaned',
                                      rep_row.get('raw_text', ''))

# Severity estimate (inlined compute_cluster_severity as it's a helper for this function)
    def compute_cluster_severity(cluster_size, source_count, credibility):
        score = (
            min(cluster_size / 50, 1.0) * 0.4 +
            min(source_count / 3,  1.0) * 0.3 +
            credibility * 0.3
        )
        if score >= 0.75:
            return 'critical'
        elif score >= 0.50:
            return 'high'
        elif score >= 0.25:
            return 'medium'
        return 'low'

    cluster_size = len(cluster_df)
    source_platforms = list(cluster_df['source_platform'].unique()) \
        if 'source_platform' in cluster_df else []
    source_count = len(source_platforms)

    credibility_mean = float(cluster_df[cred_col].mean()) \
        if cred_col in cluster_df else 0.5

    severity = compute_cluster_severity(cluster_size, source_count, credibility_mean)

    return {
        'event_type': event_type,
        'event_lat': event_lat,
        'event_lon': event_lon,
        'event_timestamp': str(event_timestamp) if event_timestamp else None,
        'event_severity': severity,
        'cluster_size': cluster_size,
        'source_count': source_count,
        'source_platforms': source_platforms,
        'cluster_credibility': round(credibility_mean, 3),
        'representative_text': str(representative_text)[:500],
        'llm_summary': None,
    }

# Try LLM summarization
use_llm = False
try:
    import yaml
    CONFIG_DIR = f'{PROJECT_DIR}/config'
    with open(f'{CONFIG_DIR}/api_keys.yaml') as f:
        keys = yaml.safe_load(f)
    GEMINI_KEY = keys.get('llm', {}).get('api_key', '')
    if GEMINI_KEY and GEMINI_KEY != 'YOUR_GEMINI_API_KEY':
        from llm_utils import generate_cluster_summary
        use_llm = True
        print(' LLM summarization enabled (Gemini)')
except Exception:
    print('LLM not configured — using extractive summaries.')

canonical_events = []
cluster_ids = sorted([c for c in df_cluster_input['cluster_id'].unique() if c != -1])

for cid in tqdm(cluster_ids, desc='Extracting events'):
    cluster_df = df_cluster_input[df_cluster_input['cluster_id'] == cid].copy()
    event = extract_canonical_event(cluster_df)
    event['event_id'] = f'EVT_{cid:04d}'

# LLM summary
    if use_llm and len(cluster_df) >= 2:
        texts = cluster_df['text_cleaned'].fillna('').tolist()
        try:
            event['llm_summary'] = generate_cluster_summary(texts, event['event_type'], GEMINI_KEY)
        except Exception:
            event['llm_summary'] = event['representative_text'][:200]
    else:
        event['llm_summary'] = event['representative_text'][:200]

    canonical_events.append(event)

# Singletons as individual events
singletons = df_cluster_input[df_cluster_input['is_singleton']].copy()
for idx, row in singletons.iterrows():
    canonical_events.append({
        'event_id': f'SINGLE_{idx}',
        'event_type': row.get('predicted_class', 'unknown'),
        'event_lat': row.get('geo_lat_resolved'),
        'event_lon': row.get('geo_lon_resolved'),
        'event_timestamp': row.get('timestamp_event_utc', row.get('timestamp_raw')),
        'event_severity': 'low',
        'cluster_size': 1,
        'source_count': 1,
        'source_platforms': [row.get('source_platform','unknown')],
        'cluster_credibility': row.get('credibility_score', 0.3),
        'representative_text': str(row.get('text_cleaned',''))[:300],
        'llm_summary': str(row.get('text_cleaned',''))[:200],
    })

df_events = pd.DataFrame(canonical_events)
print(f'\n {len(df_events)} events extracted ({len(cluster_ids)} clusters + {len(singletons)} singletons)')

Extracting events: 100%|██████████| 1/1 [00:00<00:00, 183.17it/s]


✅ 18 events extracted (1 clusters + 17 singletons)


In [12]:
# Save outputs
# clustered_events.csv — one row per real-world event
events_path = f'{DATA_PROCESSED}/clustered_events.csv'
df_events.to_csv(events_path, index=False)
print(f' Saved clustered_events.csv: {len(df_events)} events')

# master_events.csv — one row per post with cluster assignment
df_work_full = df.copy()
df_work_full = df_work_full.join(
    df_cluster_input[['cluster_id','is_singleton','credibility_score','credibility_label']],
    how='left'
)
df_work_full['cluster_id'] = df_work_full['cluster_id'].fillna(-1)
df_work_full['event_id'] = df_work_full['cluster_id'].apply(
    lambda c: f'EVT_{int(c):04d}' if c >= 0 else 'unassigned'
)

# Mark representative post per cluster
rep_ids = []
for evt in canonical_events:
    if 'representative_text' in evt:
        matching = df_work_full[
            df_work_full['text_cleaned'].str.startswith(str(evt['representative_text'])[:50])
        ]
        if not matching.empty:
            rep_ids.append(matching.index[0])
df_work_full['is_representative'] = df_work_full.index.isin(rep_ids)

master_path = f'{DATA_PROCESSED}/master_events.csv'
df_work_full.to_csv(master_path)
print(f' Saved master_events.csv: {len(df_work_full)} posts')

print(f'\n CREDIBILITY & CLUSTERING COMPLETE ')
print(f'Total real-world events found: {len(df_events)}')
print(f'Cluster events: {len(cluster_ids)}')
print(f'Singleton events: {len(singletons)}')
print(f'Mean cluster credibility: {df_events["cluster_credibility"].mean():.3f}')
print(f'\nEvent type distribution:')
print(df_events['event_type'].value_counts().to_string())
print(f'\n Next: Run 06_evaluation_and_explainability.ipynb')

✅ Saved clustered_events.csv: 18 events
✅ Saved master_events.csv: 75 posts

═══ CREDIBILITY & CLUSTERING COMPLETE ═══
Total real-world events found: 18
Cluster events: 1
Singleton events: 17
Mean cluster credibility: 0.451

Event type distribution:
event_type
earthquake    7
flood         5
other         2
accident      1
fire          1
violence      1
storm         1

👉 Next: Run 06_evaluation_and_explainability.ipynb


In [13]:
import numpy as np
from collections import Counter

try:
    from sklearn.cluster import DBSCAN
    _SKLEARN_AVAILABLE = True
except ImportError:
    _SKLEARN_AVAILABLE = False

from geo_utils import haversine_distance
from temporal_utils import temporal_proximity

# Composite Distance Matrix
def build_composite_distance_matrix(
    embeddings, timestamps, coordinates,
    weights=None
):
    """
    Build a composite pairwise distance matrix from three components:
      - semantic_dist  (1 - cosine_similarity of SBERT embeddings)
      - temporal_dist  (normalized time difference over 24h window)
      - spatial_dist   (normalized haversine distance over 100km window)

    Args:
        embeddings:    np.ndarray shape (N, D)  — SBERT vectors (L2 normalised)
        timestamps:    list of datetime or None — event timestamps
        coordinates:   list of (lat, lon) or (None, None)
        weights:       dict with keys semantic, temporal, spatial

    Returns:
        np.ndarray shape (N, N) — composite distance matrix (values 0–1)
    """
    if weights is None:
        weights = {'semantic': 0.50, 'temporal': 0.30, 'spatial': 0.20}

    N = len(embeddings)
    print(f"DEBUG: Inside build_composite_distance_matrix. N = {N}, embeddings shape = {embeddings.shape}")

# Semantic distance
# cosine similarity = dot product of unit vectors
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1e-9, norms)
    normed = embeddings / norms
    cosine_sim = normed @ normed.T
    semantic_dist = np.clip(1.0 - cosine_sim, 0.0, 1.0)

# Temporal distance
    temporal_dist = np.full((N, N), 0.5)  # neutral default if unknown
    for i in range(N):
        for j in range(i + 1, N):
            prox = temporal_proximity(timestamps[i], timestamps[j],
                                      window_hours=24)
            d = 1.0 - prox
            temporal_dist[i, j] = d
            temporal_dist[j, i] = d
    np.fill_diagonal(temporal_dist, 0.0)

# Spatial distance
    MAX_KM = 100.0
    spatial_dist = np.full((N, N), 0.5)  # neutral if geo unknown
    for i in range(N):
        for j in range(i + 1, N):
            lat1, lon1 = coordinates[i]
            lat2, lon2 = coordinates[j]
            if None in (lat1, lon1, lat2, lon2):
                d = 0.5  # unknown
            else:
                km = haversine_distance(lat1, lon1, lat2, lon2)
                d = min(1.0, km / MAX_KM)
            spatial_dist[i, j] = d
            spatial_dist[j, i] = d
    np.fill_diagonal(spatial_dist, 0.0)

# Composite
    composite = (
        weights['semantic']  * semantic_dist +
        weights['temporal']  * temporal_dist +
        weights['spatial']   * spatial_dist
    )
    return composite

# DBSCAN
def run_dbscan(distance_matrix, eps=0.35, min_samples=2):
    """
    Run DBSCAN on a precomputed distance matrix.
    Returns array of cluster labels (−1 = noise / singleton event).
    """
    if not _SKLEARN_AVAILABLE:
        print("  scikit-learn not available. Returning all as noise.")
        return np.full(len(distance_matrix), -1)

    db = DBSCAN(eps=eps, min_samples=min_samples, metric='precomputed')
    labels = db.fit_predict(distance_matrix)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    print(f"  DBSCAN: {n_clusters} clusters, {n_noise} singleton posts "
          f"(eps={eps}, min_samples={min_samples})")
    return labels

# Canonical event extraction
def extract_canonical_event(cluster_df):
    """
    Given a DataFrame of posts belonging to one cluster,
    produce a canonical event record.

    Args:
        cluster_df: DataFrame with columns including:
            post_id, predicted_class, confidence_score, credibility_score,
            geo_lat, geo_lon, geo_confidence, timestamp_event_utc,
            text_cleaned, source_platform, has_media, engagement_score

    Returns:
        dict representing the canonical event
    """
    if cluster_df.empty:
        return {}

# Event type: majority vote weighted by confidence
    event_type = cluster_df['predicted_class'].mode().iloc[0] \
        if not cluster_df['predicted_class'].isna().all() else 'unknown'

# Location: credibility-weighted centroid
    geo_rows = cluster_df.dropna(subset=['geo_lat', 'geo_lon'])
    if not geo_rows.empty:
        weights = geo_rows.get('geo_confidence', 1.0)
        if hasattr(weights, 'values'):
            weights = weights.fillna(0.5).values
        else:
            weights = np.ones(len(geo_rows))
        weights = np.where(weights == 0, 0.01, weights)
        event_lat = float(np.average(geo_rows['geo_lat'], weights=weights))
        event_lon = float(np.average(geo_rows['geo_lon'], weights=weights))
    else:
        event_lat = None
        event_lon = None

# Timestamp: median event time
    ts_col = 'timestamp_event_utc' if 'timestamp_event_utc' in cluster_df else 'timestamp_raw'
    timestamps = cluster_df[ts_col].dropna()
    event_timestamp = timestamps.median() if not timestamps.empty else None

# Representative post: highest credibility score
    cred_col = 'credibility_score' if 'credibility_score' in cluster_df.columns \
        else 'confidence_score'
    rep_row = cluster_df.loc[cluster_df[cred_col].idxmax()]
    representative_text = rep_row.get('text_cleaned',
                                      rep_row.get('raw_text', ''))

# Severity estimate
    cluster_size = len(cluster_df)
    source_platforms = list(cluster_df['source_platform'].unique()) \
        if 'source_platform' in cluster_df else []
    source_count = len(source_platforms)

    credibility_mean = float(cluster_df[cred_col].mean()) \
        if cred_col in cluster_df else 0.5

# Severity scoring
    severity = compute_cluster_severity(cluster_size, source_count,
                                        credibility_mean)

    return {
        'event_type': event_type,
        'event_lat': event_lat,
        'event_lon': event_lon,
        'event_timestamp': str(event_timestamp) if event_timestamp else None,
        'event_severity': severity,
        'cluster_size': cluster_size,
        'source_count': source_count,
        'source_platforms': source_platforms,
        'cluster_credibility': round(credibility_mean, 3),
        'representative_text': str(representative_text)[:500],
        'llm_summary': None,  # filled later by llm_utils
    }

def compute_cluster_severity(cluster_size, source_count, credibility):
    """Estimate event severity: low / medium / high / critical."""
    score = (
        min(cluster_size / 50, 1.0) * 0.4 +
        min(source_count / 3,  1.0) * 0.3 +
        credibility * 0.3
    )
    if score >= 0.75:
        return 'critical'
    elif score >= 0.50:
        return 'high'
    elif score >= 0.25:
        return 'medium'
    return 'low'

# Cluster evaluation
def compute_cluster_purity(cluster_labels, true_labels):
    """
    Compute cluster purity: fraction of posts that share the majority label
    in their cluster.
    """
    if len(cluster_labels) != len(true_labels):
        return 0.0

    total = 0
    correct = 0
    clusters = set(cluster_labels)

    for cid in clusters:
        if cid == -1:  # noise
            continue
        mask = [i for i, l in enumerate(cluster_labels) if l == cid]
        labels_in_cluster = [true_labels[i] for i in mask]
        majority_count = Counter(labels_in_cluster).most_common(1)[0][1]
        correct += majority_count
        total += len(mask)

    return correct / total if total > 0 else 0.0

The `build_composite_distance_matrix` function is shown below. The error you're seeing indicates that the `dist_matrix` is empty, which could happen if there are issues with the input data (e.g., all timestamps or coordinates are NaN, or the embeddings are somehow invalid for distance calculation) or an edge case in the function's logic that results in no distances being computed. Please review the function, paying close attention to how it handles edge cases and `NaN` values, especially in the temporal and spatial distance calculations.